In [ ]:
from modeling.utils import *
import os
import pandas as pd


In [ ]:
# 读取三个表作为变量字典
var_dict_path = "./modeling/var_dict_stat.xlsx"

# 读取三个sheet作为不同的变量字典
d_lab = pd.read_excel(var_dict_path, sheet_name=0)  # sheet=1 (索引从0开始)
d_vit = pd.read_excel(var_dict_path, sheet_name=1)  # sheet=2
d_ward_vit = pd.read_excel(var_dict_path, sheet_name=2)  # sheet=3


In [ ]:
folders = os.listdir("data/all_op_id")


In [ ]:
folder = folders[20624]
lab = pd.read_csv(os.path.join("data/all_op_id", folder, "lab_raw.csv"))
vit = pd.read_csv(os.path.join("data/all_op_id", folder, "vit_raw.csv"))
ward_vit = pd.read_csv(os.path.join("data/all_op_id", folder, "ward_vit_raw.csv"))


In [ ]:
ward_vit


In [ ]:

# 计算统计数据
lab_stats = calculate_statistics(lab.iloc[:,1:], d_lab, data_format="wide")
vit_stats = calculate_statistics(vit.iloc[:,1:], d_vit, data_format="wide")
ward_vit_stats = calculate_statistics(ward_vit.iloc[:,1:], d_ward_vit, data_format="wide")


In [ ]:
from joblib import Parallel, delayed
from tqdm import tqdm
import os
import pandas as pd

# 定义处理单个文件夹的函数
def process_folder(folder):
    # 读取当前文件夹中的数据文件
    lab = pd.read_csv(os.path.join("data/all_op_id", folder, "lab_raw.csv"))
    vit = pd.read_csv(os.path.join("data/all_op_id", folder, "vit_raw.csv"))
    ward_vit = pd.read_csv(os.path.join("data/all_op_id", folder, "ward_vit_raw.csv"))
    
    # 计算统计数据
    lab_stats = calculate_statistics(lab.iloc[:,1:], d_lab, data_format="wide")
    vit_stats = calculate_statistics(vit.iloc[:,1:], d_vit, data_format="wide")
    ward_vit_stats = calculate_statistics(ward_vit.iloc[:,1:], d_ward_vit, data_format="wide")
    
    # 添加op_id列
    lab_stats['op_id'] = folder
    vit_stats['op_id'] = folder
    ward_vit_stats['op_id'] = folder
    
    return (lab_stats, vit_stats, ward_vit_stats)

# 设置进程数
num_processes = os.cpu_count() - 1  # 留一个核心给系统

# 使用joblib进行并行处理，并显示进度条
results = Parallel(n_jobs=num_processes, verbose=10)(
    delayed(process_folder)(folder) for folder in folders
)


In [ ]:
# 合并结果
all_lab = pd.concat([result[0] for result in results], ignore_index=True)
all_vit = pd.concat([result[1] for result in results], ignore_index=True)
all_ward_vit = pd.concat([result[2] for result in results], ignore_index=True)


In [ ]:
# 修改函数以保持未找到变量的原始类型
def restore_data_types(df, var_dict):
    """
    根据变量字典恢复DataFrame中各列的数据类型，从右边第一个下划线分割列名
    对于在变量字典中未找到的变量，保持其原始数据类型
    
    参数:
        df (pd.DataFrame): 包含统计量的DataFrame
        var_dict (pd.DataFrame): 变量字典DataFrame，包含item_name和value_type列
        
    返回:
        pd.DataFrame: 恢复数据类型后的DataFrame
    """
    # 创建变量名到类型的映射
    var_type_map = dict(zip(var_dict['item_name'], var_dict['value_type']))
    
    # 复制DataFrame以避免修改原始数据
    df_typed = df.copy()
    
    # 遍历DataFrame的每一列
    for col in df.columns:
        # 从右边分割列名，分为变量名和统计量两部分
        parts = col.rsplit('_', 1)
        if len(parts) < 2:
            print(f"列 {col} 不包含下划线，保持原始类型")
            continue
            
        var_name, stat_name = parts
        
        # 如果变量名在变量字典中
        if var_name in var_type_map:
            var_type = var_type_map[var_name]
            
            # 根据变量类型设置数据类型
            if var_type in ['num', 'bin']:
                # 数值型和二元型保持为数值类型
                try:
                    df_typed[col] = pd.to_numeric(df[col], errors='coerce')
                except:
                    print(f"无法将列 {col} 转换为数值类型，保持原始类型")
            elif var_type in ['ord', 'cat']:
                # 有序型和分类型转换为分类类型
                try:
                    # 先转换为字符串，再转换为分类类型
                    df_typed[col] = df[col].astype(str).astype('category')
                except:
                    print(f"无法将列 {col} 转换为分类类型，保持原始类型")
        else:
            # 在变量字典中未找到变量名，保持原始类型
            print(f"在变量字典中未找到变量名 {var_name}，保持原始类型")
    
    return df_typed


In [ ]:
typed_all_lab = restore_data_types(all_lab, d_lab)
typed_all_vit = restore_data_types(all_vit, d_vit)
typed_all_ward_vit = restore_data_types(all_ward_vit, d_ward_vit)


In [ ]:
# 将转换好的 DataFrame 保存为 CSV 文件
output_path = "data/"

# 保存实验室检查数据
typed_all_lab.to_csv(output_path + "lab_stats.csv", index=False)
print(f"实验室检查数据已保存到: {output_path}lab_stats.csv")

# 保存生命体征数据
typed_all_vit.to_csv(output_path + "vit_stats.csv", index=False)
print(f"生命体征数据已保存到: {output_path}vit_stats.csv")

# 保存病房生命体征数据
typed_all_ward_vit.to_csv(output_path + "ward_vit_stats.csv", index=False)
print(f"病房生命体征数据已保存到: {output_path}ward_vit_stats.csv")
